<a href="https://colab.research.google.com/github/borgesjose/Grover_Algorithm/blob/main/Algoritmo__de__Grover.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Algoritmo de Grover**

Nesta atividade, são implementados os códigos do algoritmo de grover em Python, utilizando o ambiente Jupyter (collab).

Aluno: José Borges do Carmo Neto; https://github.com/borgesjose

Link do github: [Notebook Salvo](https://github.com/borgesjose/Grover_Algorithm/blob/3189437e7373dc336f2c820f24cd6223cbdad349/Algoritmo__de__Grover.ipynb)

Adotaremos a notação da base computacional $\{|00\rangle, |01\rangle, |10\rangle, |11\rangle\}$, para descrever a evolução dos circuitos passo a passo.*texto em itálico*

In [2]:
!pip install qiskit qiskit-aer -q

In [3]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

import numpy as np

## Criar  PORTA Z multicontrolada:

In [4]:
def apply_multi_controlled_z(qc, qubits):
  if len(qubits) == 1:
        qc.z(qubits[0])
        return
  controls = qubits[:-1]
  target = qubits[-1]
  qc.h(target)
  qc.mcx(controls, target)
  qc.h(target)

In [5]:
def build_oracle(n, targets):
    qc = QuantumCircuit(n, name="Oraculo")
    for t in targets:
        bits = format(t, f"0{n}b")   # ex: n=3, t=5 -> '101'
        # descobrir quais posicoes de qubit (0 a n-1) tem bit '0' no alvo,
        # lembrando da inversao little-endian (bits[n-1-i] corresponde a qubit[i])
        qubits_to_flip = []
        for i in range(n):
            if bits[n - 1 - i] == '0':
                qc.x(i)
                qubits_to_flip.append(i)

        # aplicar apply_multi_controlled_z(qc, list(range(n)))
        apply_multi_controlled_z(qc, list(range(n)))

        # desfazer os X (mesma lista de antes)
        for i in qubits_to_flip:
            qc.x(i)
    return qc

In [7]:
def build_diffuser(n):
    qc = QuantumCircuit(n, name="Difusor")
    # H em todos os qubits
    qc.h(range(n))
    # X em todos os qubits
    qc.x(range(n))
    # apply_multi_controlled_z nos n qubits
    apply_multi_controlled_z(qc, list(range(n)))
    # X em todos os qubits
    qc.x(range(n))
    # H em todos os qubits
    qc.h(range(n))

    return qc

In [8]:
import numpy as np

def num_iterations(n, k):
    N = 2 ** n
    r = np.floor(np.pi / 4 * np.sqrt(N / k))
    return int(r)

def grover_circuit(n, targets, r=None):
    k = len(targets)
    if r is None:
        r = num_iterations(n, k)
    qc = QuantumCircuit(n, n)
    # H em todos os qubits (superposicao inicial)
    qc.h(range(n))
    oracle = build_oracle(n, targets)
    diffuser = build_diffuser(n)
    for _ in range(r):
        # aplicar oracle no circuito (use qc.compose(oracle, range(n), inplace=True))
        qc.compose(oracle, range(n), inplace=True)
        # aplicar diffuser da mesma forma
        qc.compose(diffuser, range(n), inplace=True)
    # medir todos os qubits nos bits classicos correspondentes
    qc.measure(range(n), range(n))
    return qc, r

### Cenario 1


In [9]:
qc1, r1 = grover_circuit(2, [3])
print("r =", r1)

sim = AerSimulator()
result = sim.run(qc1, shots=1024).result()
counts = result.get_counts()
print(counts)

r = 1
{'11': 1024}


### Cenario 2

In [10]:
import time

def classical_linear_search(N, targets):
    targets_set = set(targets)
    t0 = time.perf_counter()
    consultas = 0
    encontrado = None
    for x in range(N):
        consultas += 1
        if x in targets_set:
            encontrado = x
            break
    t1 = time.perf_counter()
    return consultas, t1 - t0, encontrado

In [11]:
n2 = 16
alvo2 = 2**n2 - 1   # |111...1>

t0 = time.perf_counter()
qc2, r2 = grover_circuit(n2, [alvo2])
sim = AerSimulator()
result2 = sim.run(qc2, shots=1024).result()
t1 = time.perf_counter()

counts2 = result2.get_counts()
print(f"r = {r2} iteracoes")
print(f"tempo quantico (simulacao): {t1 - t0:.4f} s")
print("estado mais frequente:", max(counts2, key=counts2.get))

consultas_c, tempo_c, encontrado_c = classical_linear_search(2**n2, [alvo2])
print(f"\nbusca classica: {consultas_c} consultas, {tempo_c:.6f} s")

r = 201 iteracoes
tempo quantico (simulacao): 3.7485 s
estado mais frequente: 1111111111111111

busca classica: 65536 consultas, 0.004334 s


### Cenario 3

In [12]:
for n_teste in [16, 18, 20, 22]:
    alvo = 2**n_teste - 1
    print(f"\nTestando n={n_teste} ({2**n_teste} estados)...")
    try:
        t0 = time.perf_counter()
        qc_teste, r_teste = grover_circuit(n_teste, [alvo])
        sim = AerSimulator()
        result_teste = sim.run(qc_teste, shots=256).result()
        t1 = time.perf_counter()
        counts_teste = result_teste.get_counts()
        acerto = max(counts_teste, key=counts_teste.get) == format(alvo, f"0{n_teste}b")
        print(f"  r={r_teste} | tempo={t1-t0:.2f}s | acertou={acerto}")
    except Exception as e:
        print(f"  FALHOU: {type(e).__name__}: {e}")
        break


Testando n=16 (65536 estados)...
  r=201 | tempo=2.63s | acertou=True

Testando n=18 (262144 estados)...


KeyboardInterrupt: 

### Cenario 4

In [13]:
n4 = 5
alvos4 = [3, 7, 11]

qc4, r4 = grover_circuit(n4, alvos4)
print(f"r = {r4}")

result4 = sim.run(qc4, shots=2048).result()
counts4 = result4.get_counts()

for t in alvos4:
    b = format(t, f"0{n4}b")
    p = counts4.get(b, 0) / 2048
    print(f"|{b}> (alvo {t}): {p:.4f}")

soma_alvos = sum(counts4.get(format(t, f"0{n4}b"), 0) for t in alvos4) / 2048
print(f"\nprobabilidade total nos 3 alvos: {soma_alvos:.4f}")

r = 2
|00011> (alvo 3): 0.3276
|00111> (alvo 7): 0.3442
|01011> (alvo 11): 0.3281

probabilidade total nos 3 alvos: 1.0000


### Lidando com RUIDO

In [14]:
from qiskit_aer.noise import NoiseModel, depolarizing_error

def criar_modelo_de_ruido(p1=0.001, p2=0.01):
    noise_model = NoiseModel()
    erro_1q = depolarizing_error(p1, 1)   # taxa de erro em portas de 1 qubit
    erro_2q = depolarizing_error(p2, 2)   # taxa de erro em portas de 2 qubits
    noise_model.add_all_qubit_quantum_error(erro_1q, ["h", "x", "z"])
    noise_model.add_all_qubit_quantum_error(erro_2q, ["cx"])
    return noise_model

modelo_ruido = criar_modelo_de_ruido()

In [15]:
sim_ideal = AerSimulator()
sim_ruido = AerSimulator(noise_model=modelo_ruido)

counts_ideal = sim_ideal.run(qc4, shots=1024).result().get_counts()
counts_ruido = sim_ruido.run(qc4, shots=1024).result().get_counts()

def prob_nos_alvos(counts, n, targets, shots):
    alvos_bin = {format(t, f"0{n}b") for t in targets}
    return sum(counts.get(b, 0) for b in alvos_bin) / shots

fid_ideal = prob_nos_alvos(counts_ideal, n4, alvos4, 1024)
fid_ruido = prob_nos_alvos(counts_ruido, n4, alvos4, 1024)
print(f"fidelidade ideal: {fid_ideal:.4f}")
print(f"fidelidade com ruido: {fid_ruido:.4f}")

fidelidade ideal: 0.9990
fidelidade com ruido: 0.9590


In [16]:
lista_shots = [1, 100, 1024]

print("=== Cenario 4 (n=5) ===")
for ruido, sim_atual, nome in [(False, sim_ideal, "ideal"), (True, sim_ruido, "com ruido")]:
    for shots in lista_shots:
        counts = sim_atual.run(qc4, shots=shots).result().get_counts()
        fid = prob_nos_alvos(counts, n4, alvos4, shots)
        print(f"{nome:10s} | shots={shots:5d} | fidelidade={fid:.4f}")

=== Cenario 4 (n=5) ===
ideal      | shots=    1 | fidelidade=1.0000
ideal      | shots=  100 | fidelidade=1.0000
ideal      | shots= 1024 | fidelidade=1.0000
com ruido  | shots=    1 | fidelidade=1.0000
com ruido  | shots=  100 | fidelidade=0.9700
com ruido  | shots= 1024 | fidelidade=0.9521


In [17]:
print("\n=== Cenario 2 (n=16) - sem ruido ===")
for shots in lista_shots:
    counts = sim_ideal.run(qc2, shots=shots).result().get_counts()
    fid = prob_nos_alvos(counts, n2, [alvo2], shots)
    print(f"ideal | shots={shots:5d} | fidelidade={fid:.4f}")



=== Cenario 2 (n=16) - sem ruido ===
ideal | shots=    1 | fidelidade=1.0000
ideal | shots=  100 | fidelidade=1.0000
ideal | shots= 1024 | fidelidade=1.0000


In [18]:
print("\n=== Cenario 2 (n=16) - com ruido (cuidado, pode demorar) ===")
for shots in [1, 10, 16]:
    t0 = time.perf_counter()
    counts = sim_ruido.run(qc2, shots=shots).result().get_counts()
    t1 = time.perf_counter()
    fid = prob_nos_alvos(counts, n2, [alvo2], shots)
    print(f"com ruido | shots={shots:5d} | fidelidade={fid:.4f} | tempo={t1-t0:.2f}s")


=== Cenario 2 (n=16) - com ruido (cuidado, pode demorar) ===
com ruido | shots=    1 | fidelidade=0.0000 | tempo=2.75s
com ruido | shots=   10 | fidelidade=0.0000 | tempo=24.62s
com ruido | shots=   16 | fidelidade=0.0000 | tempo=37.81s


### Classico vs Quantico

In [19]:
def tempo_medio_iteracao_grover(n, targets, repeticoes=5):
    tempos = []
    sim = AerSimulator()
    for _ in range(repeticoes):
        qc_iter, _ = grover_circuit(n, targets, r=1)   # forca r=1: uma iteracao so
        t0 = time.perf_counter()
        sim.run(qc_iter, shots=1).result()
        t1 = time.perf_counter()
        tempos.append(t1 - t0)
    return sum(tempos) / len(tempos)

def tempo_medio_consulta_classica(repeticoes=200000):
    alvo_fake = -1  # nunca bate, forcando N comparacoes reais
    t0 = time.perf_counter()
    for x in range(repeticoes):
        _ = (x == alvo_fake)
    t1 = time.perf_counter()
    return (t1 - t0) / repeticoes

tempo_q = tempo_medio_iteracao_grover(n=10, targets=[5])
tempo_c = tempo_medio_consulta_classica()

print(f"tempo medio 1 iteracao quantica (n=10): {tempo_q*1000:.4f} ms")
print(f"tempo medio 1 consulta classica: {tempo_c*1e9:.2f} ns")

razao = tempo_q / tempo_c
print(f"1 iteracao quantica custa ~{razao:,.0f}x mais tempo de parede que 1 consulta classica")

tempo medio 1 iteracao quantica (n=10): 3.3081 ms
tempo medio 1 consulta classica: 35.26 ns
1 iteracao quantica custa ~93,813x mais tempo de parede que 1 consulta classica


In [20]:
import math

sqrt_N_cruzamento = (math.pi / 2) * razao
N_cruzamento = sqrt_N_cruzamento ** 2
n_cruzamento = math.log2(N_cruzamento)

print(f"N de cruzamento estimado: {N_cruzamento:,.0f}")
print(f"equivalente a n ~= {n_cruzamento:.1f} qubits")

N de cruzamento estimado: 21,715,363,023
equivalente a n ~= 34.3 qubits
